# AKANE E2-8：3つ目の分野（病気の分類）で「疑う網」を確かめる

問い：Gemmaの作るルールに外れが多い分野でも、ルールを疑う網は「矛盾を数えるだけ」より間違いを拾えるか。
あわせて、**どの組を確かめるかもGemmaに選ばせる**。

Gemmaへの質問の例：
- 口の答え：「Measles is a viral infection.（はしかはウイルス感染症）」は本当？
- ルール："Is it true that no cancer or other tumor is a nervous system disease?"（がんで、しかも神経の病気、というものはない？）
- 組選び：「がん」と見せて、他の21の分類から「同時にはあり得ないもの」の番号を挙げさせる

正解は研究者が作っている病気の分類表（Human Disease Ontology、2026-10-29版）。

**このノートでやること**：Gemmaに約3,800問を聞いて、答えをドライブに保存するだけ。計算はこちらで行う。
**時間の目安**（T4 GPU）：10〜20分。計画は plans/E2-8_disease.md（結果を見る前に固定）。

## 0. 準備
1. 「ランタイム」→「ランタイムのタイプを変更」→ **T4 GPU**。
2. 鍵マークの `HF_TOKEN`（前と同じもの）にアクセスを許可。
3. 「ランタイム」→「すべてのセルを実行」。

In [ ]:
# ===== 設定 =====
USE_MOCK_MOUTH = False      # True: 作り物の口（動作確認用。結果は研究に使わない）
SAVE_TO_DRIVE  = True
MODEL_ID = "google/gemma-2-2b-it"
DATA_SEED = 0

In [ ]:
import os, json, time, re, itertools, collections
import numpy as np
OUT = "/content/E2_results" if os.path.exists("/content") else "E2_results"
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/AKANE_アーキテクチャ/E2_results"
os.makedirs(OUT, exist_ok=True)
print("保存先:", OUT)

## 1. 病気のテストを作る（分類表をダウンロードして作る）

In [ ]:
# ===== 病気のテストを作る（Human Disease Ontology 2026-10-29版） =====
import os, re, json, itertools, collections, urllib.request
import numpy as np
try: from wordfreq import zipf_frequency
except ImportError:
    os.system("pip -q install wordfreq"); from wordfreq import zipf_frequency
DOID_URL = "https://raw.githubusercontent.com/DiseaseOntology/HumanDiseaseOntology/v2026-10-29/src/ontology/doid.obo"
DOID_FILE = os.environ.get("DOID_FILE", "doid_2026-10-29.obo")
if not os.path.exists(DOID_FILE): urllib.request.urlretrieve(DOID_URL, DOID_FILE)
TERMS = {}; cur = None
for line in open(DOID_FILE, encoding="utf-8"):
    line = line.rstrip("\n")
    if line == "[Term]": cur = {"isa": [], "obs": False}; continue
    if line.startswith("["): cur = None; continue
    if cur is None: continue
    if line.startswith("id: "): cur["id"] = line[4:]; TERMS[cur["id"]] = cur
    elif line.startswith("name: "): cur["name"] = line[6:]
    elif line.startswith("is_a: "): cur["isa"].append(line[6:].split(" ")[0])
    elif line.startswith("is_obsolete: true"): cur["obs"] = True
TERMS = {k: v for k, v in TERMS.items() if not v["obs"]}
assert TERMS["DOID:4"]["name"] == "disease", "分類表の読み込みに失敗"
_anc = {}
def ancestors(k):
    if k not in _anc:
        s = set()
        for p in TERMS[k]["isa"]:
            if p in TERMS: s |= {p} | ancestors(p)
        _anc[k] = s
    return _anc[k]
import sys; sys.setrecursionlimit(10000)
# ラベル22：(分類表のid, 英語の名前（冠詞なし）, 冠詞)
LABELS = {
 "cardio": ("DOID:1287", "cardiovascular system disease", "a"), "resp": ("DOID:1579", "respiratory system disease", "a"),
 "gi": ("DOID:77", "gastrointestinal system disease", "a"), "nerv": ("DOID:863", "nervous system disease", "a"),
 "skin": ("DOID:16", "skin disease", "a"), "musc": ("DOID:17", "musculoskeletal system disease", "a"),
 "urin": ("DOID:18", "urinary system disease", "a"), "endo": ("DOID:28", "endocrine system disease", "an"),
 "immune": ("DOID:2914", "immune system disease", "an"), "blood": ("DOID:74", "blood disease", "a"),
 "repro": ("DOID:15", "reproductive system disease", "a"), "breast": ("DOID:3463", "breast disease", "a"),
 "chest": ("DOID:0060118", "thoracic disease", "a"), "metab": ("DOID:0014667", "metabolic disease", "a"),
 "infect": ("DOID:0050117", "infectious disease", "an"), "tumor": ("DOID:14566", "cancer or other tumor", "a"),
 "mental": ("DOID:150", "mental disorder", "a"), "genetic": ("DOID:630", "genetic disease", "a"),
 "bact": ("DOID:104", "bacterial infection", "a"), "viral": ("DOID:934", "viral infection", "a"),
 "fungal": ("DOID:1564", "fungal infection", "a"), "parasit": ("DOID:1398", "parasitic infection", "a")}
SUB = ["bact", "viral", "fungal", "parasit"]            # 感染症の種類（どれも感染症に入る）
TOP = [l for l in LABELS if l not in SUB]
LID = {l: v[0] for l, v in LABELS.items()}
lab_of = lambda k: [l for l in LABELS if LID[l] in ancestors(k)]
# ルールの正解：分類表の全部の病気のどれかが両方を持てば「両立する」
both = collections.Counter()
for k in TERMS:
    L = set(lab_of(k)) | {l for l in LABELS if LID[l] == k}
    for p, q in itertools.combinations(sorted(L), 2): both[(p, q)] += 1
def truth_excl(p, q): return both[tuple(sorted((p, q)))] == 0      # True = 両立しない
# 病気：3語以内・よく使われる名前・数字なし・ラベルそのものではない
label_ids = set(LID.values())
cand = sorted(k for k, v in TERMS.items()
              if k not in label_ids and len(v["name"].split()) <= 3 and not re.search(r"\d", v["name"])
              and v["name"].isascii() and zipf_frequency(v["name"], "en") >= 3.0 and lab_of(k))
names = collections.Counter(TERMS[k]["name"].lower() for k in cand)
cand = [k for k in cand if names[TERMS[k]["name"].lower()] == 1]
rng = np.random.default_rng(DATA_SEED)
DIS = sorted(rng.choice(cand, min(600, len(cand)), replace=False).tolist())
STMT, GOLD, KIND, CITY = [], [], [], []
idx = {}
def add(key, text, gold, kind, city=None):
    if key in idx: return idx[key]
    idx[key] = len(STMT); STMT.append(text); GOLD.append(gold); KIND.append(kind); CITY.append(city)
    return idx[key]
nm = lambda l: LABELS[l][1]; an = lambda l: LABELS[l][2] + " " + LABELS[l][1]
# 種類→大きな分類：「ウイルス感染症は感染症の一種」（本物1つ＋偽3つ）
for s in SUB:
    fz = list(rng.choice([z for z in TOP if z != "infect"], 3, replace=False))
    for z in ["infect"] + fz:
        add(("CK", s, z), f"{an(s)[0].upper() + an(s)[1:]} is a kind of {nm(z)}.", int(z == "infect"), "種類→大分類")
# 病気→ラベル：本物のラベル全部＋偽のラベル4つ
for k in DIS:
    name = TERMS[k]["name"]; G = set(lab_of(k))
    false = [l for l in LABELS if l not in G]
    ask = sorted(G) + list(rng.choice(false, min(4, len(false)), replace=False))
    for l in ask:
        add(("CL", name, l), f"{name[0].upper() + name[1:]} is {an(l)}.", int(l in G), "病気→種類" if l in SUB else "病気→分類", name)
GOLD = np.array(GOLD)
DUMMY = len(STMT)
# つながりのルール（人が書いたもの、E2-3と同じ形）：病気がウイルス感染症で、ウイルス感染症が感染症の一種 → 病気は感染症
RULES = set()
for (t, a, b), i in list(idx.items()):
    if t == "CL" and b in SUB:
        for z in LABELS:
            if ("CK", b, z) in idx and ("CL", a, z) in idx:
                RULES.add((i, idx[("CK", b, z)], idx[("CL", a, z)]))
TRANS = sorted(RULES)
# 両立しないかどうかを確かめるラベルの組（同じ病気の問題に一緒に出てくる組）と、その組が当てはまる問題の対
INST = collections.defaultdict(list)
for name in set(c for c in CITY if c):
    L = [l for l in LABELS if ("CL", name, l) in idx]
    for p, q in itertools.combinations(sorted(L), 2): INST[(p, q)].append((idx[("CL", name, p)], idx[("CL", name, q)]))
for s in SUB:
    L = [z for z in LABELS if ("CK", s, z) in idx]
    for p, q in itertools.combinations(sorted(L), 2): INST[(p, q)].append((idx[("CK", s, p)], idx[("CK", s, q)]))
PAIRS = sorted(INST)
TRUTH = {pq: truth_excl(*pq) for pq in PAIRS}
G0 = np.append(GOLD, 0)
bad_t = sum(1 for p, q, h in TRANS if G0[p] == 1 and G0[q] == 1 and G0[h] == 0)
bad_e = sum(1 for pq in PAIRS if TRUTH[pq] for i, j in INST[pq] if G0[i] == 1 and G0[j] == 1)
print(f"病気 {len(DIS)}（候補 {len(cand)}）、問題 {len(STMT)}（本物 {GOLD.sum()}）、つながりのルール {len(TRANS)}本、"
      f"ラベルの組 {len(PAIRS)}（両立しない {sum(TRUTH.values())}）、正解が破るルール {bad_t}+{bad_e}")
print("例:", " | ".join(STMT[i] for i in [0, 1, 20, 21, 22, 23]))

## 2. 質問を作る
- 口の答え：全部の問題（"Is the following statement true?"、前と同じ聞き方）
- ルール：ラベルの組231組 × 順番2通り（"Is it true that no X is a Y?"、はい＝両立しない）
- 組選び：ラベル22個それぞれ（Gemmaに番号を書かせる）

In [ ]:
MOUTH_Q = [f"Is the following statement true? Answer Yes or No only.\nStatement: {s}" for s in STMT]
RULE_QS, RULE_Q = [], []
for j, (p, q) in enumerate(PAIRS):
    for o, (x, y) in enumerate([(p, q), (q, p)]):
        RULE_QS.append((j, o)); RULE_Q.append(f"Is it true that no {nm(x)} is {an(y)}? Answer Yes or No only.")
CHOOSE = []
for x in sorted(LABELS):
    opts = [y for y in sorted(LABELS) if y != x]
    lst = "\n".join(f"{n + 1}. {nm(y)}" for n, y in enumerate(opts))
    CHOOSE.append((x, opts, f"Here is a list of disease categories:\n{lst}\n\n"
                   f"Which of these categories can never apply to the same disease as \"{nm(x)}\"? "
                   f"That is, no disease can be both {an(x)} and that category. Answer with the numbers only, separated by commas."))
print(f"口 {len(MOUTH_Q)}問、ルール {len(RULE_Q)}問、組選び {len(CHOOSE)}問")
print(MOUTH_Q[22]); print(RULE_Q[0]); print(CHOOSE[0][2][:200], "…")

## 3. Gemmaに聞く（保存するので2回目からは飛ばせる）

In [ ]:
cache = f"{OUT}/E2-8_answers_{'MOCK' if USE_MOCK_MOUTH else MODEL_ID.split('/')[-1]}.json"
def parse_pick(text, opts):
    nums = [int(n) for n in re.findall(r"\d+", text)]
    return sorted({opts[n - 1] for n in nums if 1 <= n <= len(opts)})
if os.path.exists(cache):
    d = json.load(open(cache))
    if d["stmt"] != STMT or d["rule_q"] != RULE_Q: raise SystemExit("保存済みの答えと問題が合わない。ファイルを消してやり直す")
    print("保存済みの答えを読み込んだ:", cache)
else:
    if USE_MOCK_MOUTH:   # 作り物の口：間違い12%、ルールは「両立しない」寄り
        r = np.random.default_rng(123)
        ML = 2.0 * (2 * GOLD - 1) + r.normal(0, 1.5, len(GOLD))
        fl = r.random(len(GOLD)) < 0.12; ML[fl] = -np.sign(ML[fl]) * np.abs(r.normal(2, 0.7, fl.sum()))
        tr = np.array([TRUTH[PAIRS[j]] for j, o in RULE_QS])
        RL = 1.5 * (2 * tr - 1) + 1.0 + r.normal(0, 1.5, len(tr))
        texts = [", ".join(str(n + 1) for n, y in enumerate(o) if (truth_excl(x, y) and r.random() < .6) or r.random() < .1) for x, o, q in CHOOSE]
    else:
        import torch
        from transformers import AutoTokenizer, AutoModelForCausalLM
        from google.colab import userdata
        tok = AutoTokenizer.from_pretrained(MODEL_ID, token=userdata.get("HF_TOKEN"), padding_side="left")
        model = AutoModelForCausalLM.from_pretrained(MODEL_ID, token=userdata.get("HF_TOKEN"),
                                                     torch_dtype=torch.float16, device_map="cuda").eval()
        YES = tok.encode("Yes", add_special_tokens=False)[0]; NO = tok.encode("No", add_special_tokens=False)[0]
        pr = lambda s: tok.apply_chat_template([{"role": "user", "content": s}], tokenize=False, add_generation_prompt=True)
        def yesno(Q, B=64):
            L = np.zeros(len(Q)); t = time.time()
            for i in range(0, len(Q), B):
                enc = tok([pr(q) for q in Q[i:i + B]], return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
                with torch.no_grad(): last = model(**enc).logits[:, -1, :].float()
                L[i:i + B] = (last[:, YES] - last[:, NO]).cpu().numpy()
                if (i // B) % 20 == 0: print(f"  {i + B}/{len(Q)}  {time.time() - t:.0f}秒", flush=True)
            return L
        print("口の答え"); ML = yesno(MOUTH_Q)
        print("ルール"); RL = yesno(RULE_Q)
        print("組選び"); texts = []
        for x, o, q in CHOOSE:
            enc = tok([pr(q)], return_tensors="pt", add_special_tokens=False).to("cuda")
            with torch.no_grad(): g = model.generate(**enc, max_new_tokens=80, do_sample=False)
            texts.append(tok.decode(g[0, enc["input_ids"].shape[1]:], skip_special_tokens=True))
    d = {"model": "MOCK" if USE_MOCK_MOUTH else MODEL_ID, "stmt": STMT, "mouth": ML.tolist(),
         "rule_q": RULE_Q, "rule_qs": RULE_QS, "rule_logit": RL.tolist(),
         "pairs": PAIRS, "truth": [TRUTH[p] for p in PAIRS],
         "choose": [{"label": x, "options": o, "text": t, "picked": parse_pick(t, o)} for (x, o, q), t in zip(CHOOSE, texts)]}
    json.dump(d, open(cache, "w"), ensure_ascii=False); print("保存:", cache)

## 4. ざっと確かめる（ここまでで終わり。結果のファイルをこちらで読みます）

In [ ]:
ML = np.array(d["mouth"]); print(f"口の正解率 {((ML > 0) == GOLD).mean():.3f}（問題 {len(GOLD)}）")
v = np.zeros((len(PAIRS), 2))
for (j, o), l in zip(d["rule_qs"], d["rule_logit"]): v[j, o] = l
g = v.mean(1) > 0; tt = np.array(d["truth"])
print(f"ルール（全部の組）：正しい判断 {(g == tt).mean():.1%}、間違ったルール {(~tt & g).sum()}/{(~tt).sum()}、見逃し {(tt & ~g).sum()}/{tt.sum()}")
ch = {tuple(sorted((c["label"], y))) for c in d["choose"] for y in c["picked"]}
cw = sum(not truth_excl(*p) for p in ch)
print(f"Gemmaが選んだ組 {len(ch)}、そのうち実は両立する組 {cw}")
for c in d["choose"][:3]: print(" ", nm(c["label"]), "→", c["text"][:80].replace(chr(10), " "))